In [0]:
sales_df = spark.table(
    "retail_medallion_ws.default.bronze_sales_details"
)


In [0]:
sales_df.printSchema()

root
 |-- sls_ord_num: string (nullable = true)
 |-- sls_prd_key: string (nullable = true)
 |-- sls_cust_id: integer (nullable = true)
 |-- sls_order_dt: integer (nullable = true)
 |-- sls_ship_dt: integer (nullable = true)
 |-- sls_due_dt: integer (nullable = true)
 |-- sls_sales: integer (nullable = true)
 |-- sls_quantity: integer (nullable = true)
 |-- sls_price: integer (nullable = true)



In [0]:
display(sales_df.limit(10))

sls_ord_num,sls_prd_key,sls_cust_id,sls_order_dt,sls_ship_dt,sls_due_dt,sls_sales,sls_quantity,sls_price
SO43697,BK-R93R-62,21768,20101229,20110105,20110110,3578,1,3578
SO43698,BK-M82S-44,28389,20101229,20110105,20110110,3400,1,3400
SO43699,BK-M82S-44,25863,20101229,20110105,20110110,3400,1,3400
SO43700,BK-R50B-62,14501,20101229,20110105,20110110,699,1,699
SO43701,BK-M82S-44,11003,20101229,20110105,20110110,3400,1,3400
SO43702,BK-R93R-44,27645,20101230,20110106,20110111,3578,1,3578
SO43703,BK-R93R-62,16624,20101230,20110106,20110111,3578,1,3578
SO43704,BK-M82B-48,11005,20101230,20110106,20110111,3375,1,3375
SO43705,BK-M82S-38,11011,20101230,20110106,20110111,3400,1,3400
SO43706,BK-R93R-48,27621,20101231,20110107,20110112,3578,1,3578


In [0]:
%sql

SELECT COUNT(*) AS total_rows
FROM retail_medallion_ws.default.bronze_sales_details;

total_rows
60398


In [0]:
%sql

SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN sls_ord_num IS NULL THEN 1 ELSE 0 END) AS null_sls_ord_num,
    SUM(CASE WHEN sls_prd_key IS NULL THEN 1 ELSE 0 END) AS null_sls_prd_key,
    SUM(CASE WHEN sls_cust_id IS NULL THEN 1 ELSE 0 END) AS null_sls_cust_id,
    SUM(CASE WHEN sls_order_dt IS NULL THEN 1 ELSE 0 END) AS null_sls_order_dt,
    SUM(CASE WHEN sls_ship_dt IS NULL THEN 1 ELSE 0 END) AS null_sls_ship_dt,
    SUM(CASE WHEN sls_due_dt IS NULL THEN 1 ELSE 0 END) AS null_sls_due_dt,
    SUM(CASE WHEN sls_sales IS NULL THEN 1 ELSE 0 END) AS null_sls_sales,
    SUM(CASE WHEN sls_quantity IS NULL THEN 1 ELSE 0 END) AS null_sls_quantity,
    SUM(CASE WHEN sls_price IS NULL THEN 1 ELSE 0 END) AS null_sls_price
FROM retail_medallion_ws.default.bronze_sales_details;

total_rows,null_sls_ord_num,null_sls_prd_key,null_sls_cust_id,null_sls_order_dt,null_sls_ship_dt,null_sls_due_dt,null_sls_sales,null_sls_quantity,null_sls_price
60398,0,0,0,0,0,0,8,0,7


In [0]:
%sql

SELECT
    sls_ord_num,
    sls_prd_key,
    COUNT(*) AS record_count
FROM retail_medallion_ws.default.bronze_sales_details
GROUP BY
    sls_ord_num,
    sls_prd_key
HAVING COUNT(*) > 1;

sls_ord_num,sls_prd_key,record_count


In [0]:
%sql

SELECT *
FROM retail_medallion_ws.default.bronze_sales_details
WHERE sls_sales IS NULL;

sls_ord_num,sls_prd_key,sls_cust_id,sls_order_dt,sls_ship_dt,sls_due_dt,sls_sales,sls_quantity,sls_price
SO61548,CA-1098,12386,20130705,20130712,20130717,null,1,9
SO61560,HL-U509,16213,20130705,20130712,20130717,null,1,35
SO61561,CL-9009,18380,20130705,20130712,20130717,null,1,8
SO61562,FE-6654,18944,20130705,20130712,20130717,null,1,22
SO67551,BC-M005,15150,20131001,20131008,20131013,null,1,10
SO68588,PK-7098,11318,20131017,20131024,20131029,null,1,2
SO68745,HL-U509,18262,20131020,20131027,20131101,null,1,35
SO68745,GL-H102-L,18262,20131020,20131027,20131101,null,1,24


In [0]:
%sql

CREATE OR REPLACE TABLE retail_medallion_ws.default.silver_sales AS

SELECT
    sls_ord_num,
    sls_prd_key,
    sls_cust_id,
    sls_order_dt,
    sls_ship_dt,
    sls_due_dt,
    sls_sales,
    sls_quantity,
    sls_price
FROM retail_medallion_ws.default.bronze_sales_details
WHERE sls_sales IS NOT NULL
  AND sls_price IS NOT NULL;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN sls_sales IS NULL THEN 1 ELSE 0 END) AS null_sales,
    SUM(CASE WHEN sls_price IS NULL THEN 1 ELSE 0 END) AS null_price
FROM retail_medallion_ws.default.silver_sales;

total_rows,null_sales,null_price
60383,0,0
